In [1]:
import polars as pl
from pathlib import Path
import numpy as np
import pyboat
from pyboat import WAnalyzer
from pyboat import plotting
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import pandas as pd
from scipy.signal import savgol_filter

dataset = Path("/home/derfelt/PartchLabFiles/Circadian "
                    "Rhythms/Live Study/enVision/enVision_pre"
                    "processed/2026-05-11._.PREPROCESSED.parquet")

detected_period = Path("/home/derfelt/PartchLabFiles/Circadian "
                       "Rhythms/Live Study/enVision/period_dete"
                       "ction_results/5-11_per-well_period_results.csv")

lf = pl.scan_parquet(dataset).filter(pl.col("feature") == "CalcResultI")

tpoints = lf.select("Timepoint").unique().sort(by="Timepoint").collect().to_series().to_numpy()

detected_period = pl.read_csv(detected_period)
# display(detected_period)

platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

platemap = detected_period.join(platemap, on="Well", how='left')

print(len(tpoints))
display(lf.head(2).collect())
display(platemap)

206


Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",30.0,"""CalcResultI""",15.122279,15.122279,null,15.17334,15.163128,15.14781,15.122279,15.096748,15.08143,15.071217
"""A01""",30.5,"""CalcResultI""",-4.79441,-4.79441,null,-4.740661,-4.751411,-4.767536,-4.79441,-4.821285,-4.837409,-4.848159


Well,group_label,detected_period,MoleculeID,strain,sample_type
str,str,f64,str,str,str
"""G06""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""G07""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""G05""","""0.25% DMSO | bmal1""",29.727273,"""0.25% DMSO""","""bmal1""","""dex"""
"""L05""","""0.25% DMSO | per2""",25.153846,"""0.25% DMSO""","""per2""","""dex"""
"""L06""","""0.25% DMSO | per2""",25.153846,"""0.25% DMSO""","""per2""","""dex"""
…,…,…,…,…,…
"""O21""","""nan | nan""",25.153846,null,null,null
"""I03""","""nan | nan""",46.714286,null,null,null
"""N17""","""nan | nan""",23.357143,null,null,null


In [2]:
def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

In [ ]:
sampling_interval = 0.5 # hours
period_range = (18,40)
n_periods = 200
ridge_thresh = 2
timepoint_col = "Timepoint"

stat_val = "mean"

well_col = "Well"
wells = lf.select(well_col).unique().collect().sort("Well").to_series().to_list()
print(wells)

win_len=13
polyorder=3
detrended_data = lf
periods = np.linspace(period_range[0], period_range[1],n_periods)


out_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/wavelet_analysis")

results = []

out_pdf = out_path / "wavelet_analysis_visuals.pdf"
with PdfPages(out_pdf) as pdf:
    for well in tqdm(wells, desc="Wells"):
        well_df = (detrended_data
                   .filter(pl.col(well_col) == well)
                   .sort('Timepoint'))
        
        
        signal = well_df.select(stat_val).collect().to_series().to_numpy()
        
        group = platemap.filter(pl.col(well_col) == well).select('group_label').item()
        
        # Wavelet analysis
        wan = WAnalyzer(periods=periods, dt=sampling_interval, time_unit_label='h')
        
        # wavelet spectrum
        wan.compute_spectrum(signal, do_plot=False)
        
        ridge = wan.get_maxRidge(power_thresh=ridge_thresh)
        # should contain time, period, phase, amplitude, power
        ridge['time'] = tpoints
        
        fourier_power = wan.get_averaged_spectrum()
        
        # display(ridge)
        # print(fourier_power)
        
        if ridge is not None and not ridge.empty:
            mean_period = ridge['periods'].mean()
            std_period = ridge['periods'].std()
            median_period = ridge['periods'].median()
            
            mean_amplitude = ridge['amplitude'].mean()
            std_amplitude = ridge['amplitude'].std()
            median_amplitude = ridge['amplitude'].median()
            
            mid_idx = len(ridge) // 2
            mid_phase = ridge['phase'].iloc[mid_idx]
            
            results.append({
                'Well': well,
                'group_label': group,
                'mean_period': mean_period,
                'std_period': std_period,
                'median_period': median_period,
                'mean_amplitude': mean_amplitude,
                'std_amplitude': std_amplitude,
                'median_amplitude': median_amplitude,
                'mid_phase': mid_phase
            })
            
        # plots
        fig, axs = plt.subplots(3,1,figsize=(14,12))
        
        # plot detrended signal
        signal = savgol_filter(signal, window_length=win_len, polyorder=polyorder)
        time_axis = tpoints
        axs[0].plot(time_axis, signal, color='royalblue', linewidth=1.5)
        axs[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
        axs[0].set_xlabel('Time (hours)')
        axs[0].set_ylabel('Expression')
        axs[0].set_title('Detrended Signal')
        # add_period_marks(ax=axs[0], x_min=np.min(tpoints), x_max=np.max(tpoints))
        
        # wavelet power specturm
        extent = [time_axis[0], time_axis[-1], periods[-1], periods[0]]
        
        axs[1].imshow(np.abs(wan.transform), aspect='auto', extent=extent, cmap='viridis', origin='upper')
        if ridge is not None and not ridge.empty:
            axs[1].plot(ridge['time'], ridge['periods'], color='cyan', linewidth=2, label='ridge')
        axs[1].set_xlabel('Time (hours)')
        axs[1].set_ylabel('Period (hours)')
        axs[1].set_title('Wavelet Power Spectrum')
        axs[1].legend(fontsize=8)
        # add_period_marks(ax=axs[1], x_min=np.min(tpoints), x_max=np.max(tpoints))
        
        # fourier spectrum
        axs[2].plot(periods, fourier_power, color='darkred', linewidth=1.5)
        peak_period = periods[np.argmax(fourier_power)]
        axs[2].axvline(peak_period, color='red', linewidth=1.5, linestyle='--', label=f"Peak: {peak_period:.1f}h")
        axs[2].set_xlabel('Period (hours)')
        axs[2].set_ylabel('Power')
        axs[2].set_title('Fourier Spectrum (time-averaged)')
        axs[2].legend(fontsize=8)
        # add_period_marks(ax=axs[2], x_min=np.min(tpoints), x_max=np.max(tpoints))
        
        fig.suptitle(f"Well: {well} - {group}", fontsize=11)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

# summary df

out_results = out_path / "wavelet_analysis_results.csv"
results_df = pl.DataFrame(results)
display(results_df)
results_df.write_csv(out_results)

['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B10', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D21', 'D22', 'D23', 'D24', 'E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23'

Wells: 100%|██████████| 384/384 [01:07<00:00,  5.65it/s]


Well,group_label,mean_period,std_period,median_period,mean_amplitude,std_amplitude,median_amplitude,mid_phase
str,str,f64,f64,f64,f64,f64,f64,f64
"""A01""","""nan | nan""",21.062741,2.889483,21.371859,2.218343,0.543402,2.2615,2.56414
"""A02""","""nan | nan""",22.315851,3.613964,24.190955,2.076786,0.321869,2.063742,2.244114
"""A03""","""nan | nan""",23.151973,0.349203,23.19598,3.818327,1.005645,3.934249,1.566687
"""A04""","""nan | nan""",21.22213,2.276625,21.592965,3.294639,0.489149,3.20576,2.841953
"""A05""","""nan | nan""",21.538225,3.57518,22.366834,2.302105,0.510055,2.284929,2.961322
…,…,…,…,…,…,…,…,…
"""P20""","""nan | nan""",28.902327,6.987078,24.743719,2.94051,0.747646,3.04389,1.397282
"""P21""","""nan | nan""",22.572913,4.319796,19.98995,2.436091,0.660807,2.454722,0.28664
"""P22""","""nan | nan""",23.662877,5.486553,19.547739,2.411325,0.985024,2.269835,4.447099
